# Daisyworld — сравнительный анализ динамики численности

## Введение

В этом документе мы проведём сравнительный анализ динамики численности
маргариток для различных наборов параметров. Это позволит наглядно
увидеть, как начальные условия влияют на популяционную динамику.

## Подготовка окружения

Загружаем необходимые пакеты и активируем проект:

In [1]:
using DrWatson
@quickactivate "project"
using Agents
using DataFrames
using Plots
include(srcdir("daisyworld.jl"))
using CairoMakie

## Определение функций для подсчёта маргариток

In [2]:
black(a) = a.breed == :black
white(a) = a.breed == :white
adata = [(black, count), (white, count)]

2-element Vector{Tuple{Function, typeof(count)}}:
 (Main.black, count)
 (Main.white, count)

## Параметры для сравнительного анализа

Задаём набор параметров для сравнения. Мы варьируем `max_age` и
`init_white`:

In [3]:
param_list = [
    (max_age = 25, init_white = 0.2, init_black = 0.2),
    (max_age = 40, init_white = 0.2, init_black = 0.2),
    (max_age = 25, init_white = 0.8, init_black = 0.2),
    (max_age = 40, init_white = 0.8, init_black = 0.2),
]

4-element Vector{@NamedTuple{max_age::Int64, init_white::Float64, init_black::Float64}}:
 (max_age = 25, init_white = 0.2, init_black = 0.2)
 (max_age = 40, init_white = 0.2, init_black = 0.2)
 (max_age = 25, init_white = 0.8, init_black = 0.2)
 (max_age = 40, init_white = 0.8, init_black = 0.2)

## Сбор данных для всех комбинаций параметров

Запускаем симуляции для каждого набора параметров и сохраняем
результаты:

In [4]:
results = []
for (i, params) in enumerate(param_list)
    model = daisyworld(; 
        griddims = (30, 30),
        albedo_white = 0.75,
        albedo_black = 0.25,
        surface_albedo = 0.4,
        solar_change = 0.005,
        solar_luminosity = 1.0,
        scenario = :default,
        seed = 165,
        params...
    )
    agent_df, _ = run!(model, 1000; adata)
    push!(results, (params = params, df = agent_df, idx = i))
end

## Построение сравнительного графика

Создаём фигуру для сравнительного анализа:

In [5]:
figure = Figure(size = (800, 500))
ax = Axis(figure[1, 1], 
    xlabel = "tick", 
    ylabel = "daisy count",
    title = "Сравнение динамики численности для разных параметров"
)

Axis with 0 plots:

### Цветовая схема для разных параметров

In [6]:
colors = [:blue, :green, :orange, :red]
markers = [:circle, :square, :diamond, :utriangle]

4-element Vector{Symbol}:
 :circle
 :square
 :diamond
 :utriangle

### Построение линий для каждого набора параметров

In [7]:
for (i, result) in enumerate(results)
    params = result.params
    df = result.df
    label = "max_age=$(params.max_age), init_white=$(params.init_white)"
    lines!(ax, df[!, :time], df[!, :count_black], 
        color = colors[i], linestyle = :solid, label = "black: $label")
    lines!(ax, df[!, :time], df[!, :count_white], 
        color = colors[i], linestyle = :dash, label = "white: $label")
end

### Добавление легенды

In [8]:
Legend(figure[1, 2], ax, "Параметры", labelsize = 10)

Legend()

## Сохранение результата

In [9]:
save(plotsdir("daisy_count_comparison.png"), figure)

## Анализ результатов

На полученном графике можно наблюдать:

-   **Сплошные линии** — численность чёрных маргариток
-   **Пунктирные линии** — численность белых маргариток
-   **Разные цвета** — разные комбинации параметров

Это позволяет сравнить, как: - Увеличение `max_age` влияет на
устойчивость популяции - Увеличение `init_white` влияет на начальную
динамику и равновесие

## Заключение

Сравнительный анализ показывает чувствительность модели к начальным
параметрам. Различные комбинации `max_age` и `init_white` приводят к
разным траекториям эволюции системы.